<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/04_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.7.0?labpath=examples%2F04_export.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/04_export.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>

# 04 — Framework Export

Export circuits to all **8 supported frameworks**: OpenQASM 3.0, Qiskit, PennyLane, Cirq, TKET, Amazon Braket, Q#, IQM.
Frameworks without the optional dependency installed are skipped automatically.

```bash
pip install quprep[qiskit]
pip install quprep[pennylane]
pip install quprep[cirq]
pip install quprep[tket]
pip install quprep[braket]
pip install quprep[qsharp]   # Azure Quantum submission
pip install quprep[iqm]      # IQM hardware submission
```

In [ ]:
import json

import numpy as np

import quprep as qd
from quprep.encode.zz_feature_map import ZZFeatureMapEncoder
from quprep.export.iqm_export import IQMExporter
from quprep.export.qsharp_export import QSharpExporter

rng = np.random.default_rng(42)
X = rng.uniform(0, 1, size=(5, 3))

enc = qd.AngleEncoder(rotation="ry").encode(np.array([0.3, 1.1, 0.7]) * np.pi)
enc_zz = ZZFeatureMapEncoder(reps=1).encode(np.array([0.5, 1.2, 0.8]))

## 1. OpenQASM 3.0 — no dependencies

Universal interchange format accepted by all major frameworks and hardware backends.

In [ ]:
exp = qd.QASMExporter()
print(exp.export(enc))

## 2. Batch save — write each sample to its own file

In [ ]:
result = qd.prepare(X, encoding="angle")
paths = exp.save_batch(result.encoded, "/tmp/quprep_batch/", stem="circuit")

print(f"Saved {len(paths)} files to {paths[0].parent}/")
for p in paths:
    print(f"  {p.name}")

## 3. Qiskit

Returns a `qiskit.QuantumCircuit`. Requires `pip install quprep[qiskit]`.

In [ ]:
try:
    from quprep.export.qiskit_export import QiskitExporter

    qc = QiskitExporter().export(enc)
    print(qc.draw(output="text"))
    print(f"num_qubits : {qc.num_qubits}")
    print(f"depth      : {qc.depth()}")
except ImportError as e:
    print(f"skipped — {e}")

## 4. Amplitude encoding via Qiskit (StatePreparation)

In [ ]:
try:
    from quprep.export.qiskit_export import QiskitExporter

    unit_vec = np.array([0.5, 0.5, 0.5, 0.5])
    enc_amp = qd.AmplitudeEncoder().encode(unit_vec)
    qc_amp = QiskitExporter().export(enc_amp)
    print(qc_amp.draw(output="text"))
except ImportError as e:
    print(f"skipped — {e}")

## 5. PennyLane

Returns a callable `qml.QNode`. Requires `pip install quprep[pennylane]`.

In [ ]:
try:
    from quprep.export.pennylane_export import PennyLaneExporter

    qnode = PennyLaneExporter().export(enc)
    state = qnode()
    print(f"QNode  : {qnode}")
    print(f"State  : {state}")
except ImportError as e:
    print(f"skipped — {e}")

## 6. Cirq

Returns a `cirq.Circuit`. Requires `pip install quprep[cirq]`.

In [ ]:
try:
    from quprep.export.cirq_export import CirqExporter

    circuit = CirqExporter().export(enc)
    print(circuit)
    print(f"qubits : {sorted(circuit.all_qubits())}")
except ImportError as e:
    print(f"skipped — {e}")

## 7. TKET / pytket

Returns a `pytket.Circuit`. Requires `pip install quprep[tket]`.

In [ ]:
try:
    from quprep.export.tket_export import TKETExporter

    circuit = TKETExporter().export(enc)
    print(f"n_qubits : {circuit.n_qubits}")
    print(f"n_gates  : {circuit.n_gates}")
    print(f"depth    : {circuit.depth()}")
except ImportError as e:
    print(f"skipped — {e}")

## 8. Amazon Braket

Returns a `braket.circuits.Circuit`. Requires `pip install quprep[braket]`.

In [ ]:
try:
    from braket.circuits import Circuit  # noqa: F401

    from quprep.export.braket_export import BraketExporter

    exp_bk = BraketExporter()

    print("Angle encoding:")
    print(exp_bk.export(enc))

    print("ZZ feature map:")
    print(exp_bk.export(enc_zz))

    result2 = qd.prepare(X[:2], encoding="angle", framework="braket")
    print(f"prepare() → braket: {type(result2.circuits[0])}")
except ImportError as e:
    print(f"  skipped — {e}")

## 9. Q# (Microsoft Azure Quantum)

Returns a `str` of Q# 1.0 source. No deps needed for generation; `pip install quprep[qsharp]` for actual Azure Quantum submission.

In [ ]:
exp_qs = QSharpExporter(namespace="QuPrepDemo", operation_name="EncodeFeatures")

qsharp_src = exp_qs.export(enc)
print("Angle encoding → Q# source:")
print(qsharp_src)

qsharp_zz = exp_qs.export(enc_zz)
print("ZZ feature map (first 6 lines):")
for line in qsharp_zz.splitlines()[:6]:
    print(line)
print("  ...")

result_qs = qd.prepare(X[:2], encoding="angle", framework="qsharp")
print(f"prepare() → qsharp: str = {isinstance(result_qs.circuits[0], str)}")

## 10. IQM Native Format

Returns a JSON-serialisable `dict` (PRX + CZ gate set). No deps needed for generation; `pip install quprep[iqm]` for hardware submission via `iqm_client`.

In [ ]:
exp_iqm = IQMExporter(circuit_name="feature_map", qubit_prefix="QB")

circuit_iqm = exp_iqm.export(enc)
print("Angle encoding (IQM dict):")
print(json.dumps(circuit_iqm, indent=2))

circuit_iqm_zz = exp_iqm.export(enc_zz)
gate_names = [op["name"] for op in circuit_iqm_zz["instructions"]]
print(f"ZZ feature map: {len(circuit_iqm_zz['instructions'])} instructions")
print(f"  gate types : {sorted(set(gate_names))}")
print(f"  CZ count   : {gate_names.count('cz')}")

result_iqm = qd.prepare(X[:2], encoding="angle", framework="iqm")
print(f"prepare() → iqm: dict = {isinstance(result_iqm.circuits[0], dict)}")